<a href="https://colab.research.google.com/github/abhi-chandra/SHAP/blob/main/shap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping

import shap
import warnings
import logging

warnings.filterwarnings("ignore")
logger = logging.getLogger('shap')
logger.disabled = True


class ExplainAnomaliesUsingSHAP:


    autoencoder = None
    num_anomalies_to_explain = None
    reconstruction_error_percent = None
    shap_values_selection = None
    counter = None

    def __init__(self, num_anomalies_to_explain=100, reconstruction_error_percent=0.5, shap_values_selection='mean'):


        self.num_anomalies_to_explain = num_anomalies_to_explain
        self.reconstruction_error_percent = reconstruction_error_percent
        self.shap_values_selection = shap_values_selection

    def train_model(self, x_train, nb_epoch=1000, batch_size=64):


        input_dim = x_train.shape[1]
        input_layer = Input(shape=(input_dim,))

        encoder = Dense(input_dim, activation="relu")(input_layer)  # Wider first layer
        encoder = Dense(int(input_dim / 2), activation="relu")(encoder)
        encoder = Dense(int(input_dim / 4), activation="relu")(encoder) # Additional layer



        decoded = Dense(input_dim, activation='sigmoid')(encoder)


        self.autoencoder = Model(inputs=input_layer, outputs=decoded)

        self.autoencoder.summary()

        self.autoencoder.compile(optimizer='adam', loss='mean_squared_error', metrics=['mse'])

        earlystopper = EarlyStopping(monitor='val_loss', patience=5, verbose=1)
        self.autoencoder.fit(x_train, x_train, epochs=nb_epoch, batch_size=batch_size, shuffle=True,
                             validation_split=0.1, verbose=2, callbacks=[earlystopper])

        return self.autoencoder

    def get_top_anomaly_to_explain(self, x_explain):


        predictions = self.autoencoder.predict(x_explain)
        square_errors = np.power(x_explain - predictions, 2)
        mse_series = pd.Series(np.mean(square_errors, axis=1))

        most_anomal_trx = mse_series.sort_values(ascending=False)
        columns = ["id", "mse_all_columns"]
        columns.extend(["squared_error_" + x for x in list(x_explain.columns)])
        items = []
        for x in most_anomal_trx.iteritems():
            item = [x[0], x[1]]
            item.extend(square_errors.loc[x[0]])
            items.append(item)

        df_anomalies = pd.DataFrame(items, columns=columns)
        df_anomalies.set_index('id', inplace=True)

        top_anomalies_to_explain = df_anomalies.head(self.num_anomalies_to_explain).index
        return top_anomalies_to_explain

    def get_num_features_with_highest_reconstruction_error(self, total_squared_error, errors_df):


        error = 0
        for num_of_features, index in enumerate(errors_df.index):
            error += errors_df.loc[index, 'err']
            if error >= self.reconstruction_error_percent * total_squared_error:
                break
        return num_of_features + 1

    def get_background_set(self, x_train, background_size=20000):


        background_set = x_train.head(background_size)
        return background_set

    def get_errors_df_per_record(self, record):


        prediction = self.autoencoder.predict(np.array([[record]])[0])[0]
        square_errors = np.power(record - prediction, 2)
        errors_df = pd.DataFrame({'col_name': square_errors.index, 'err': square_errors}).reset_index(drop=True)
        total_mse = np.mean(square_errors)
        errors_df.sort_values(by='err', ascending=False, inplace=True)
        return errors_df, total_mse

    def get_highest_shap_values(self, shap_values_df):


        all_explaining_features_df = pd.DataFrame()

        for i in range(shap_values_df.shape[0]):
            shap_values = shap_values_df.iloc[i]

            if self.shap_values_selection == 'mean':
                treshold_val = np.mean(shap_values)

            elif self.shap_values_selection == 'median':
                treshold_val = np.median(shap_values)

            elif self.shap_values_selection == 'constant':
                num_explaining_features = 5
                explaining_features = shap_values_df[i:i + 1].stack().nlargest(num_explaining_features)
                all_explaining_features_df = pd.concat([all_explaining_features_df, explaining_features], axis=0)
                continue

            else:
                raise ValueError('unknown SHAP value selection method')

            num_explaining_features = 0
            for j in range(len(shap_values)):
                if shap_values[j] > treshold_val:
                    num_explaining_features += 1
            explaining_features = shap_values_df[i:i + 1].stack().nlargest(num_explaining_features)
            all_explaining_features_df = pd.concat([all_explaining_features_df, explaining_features], axis=0)
        return all_explaining_features_df

    def func_predict_feature(self, record):

        record_prediction = self.autoencoder.predict(record)[:, self.counter]
        # print(record_prediction)
        return record_prediction

    def explain_unsupervised_data(self, x_train, x_explain, autoencoder=None, return_shap_values=False):


        self.autoencoder = autoencoder
        if self.autoencoder is None:
            self.train_model(x_train)

        top_records_to_explain = self.get_top_anomaly_to_explain(x_explain)
        all_sets_explaining_features = {}

        for record_idx in top_records_to_explain:
            print(record_idx)

            record_to_explain = x_explain.loc[record_idx]

            df_err, total_mse = self.get_errors_df_per_record(record_to_explain)
            num_of_features = self.get_num_features_with_highest_reconstruction_error(total_mse * df_err.shape[0],
                                                                                      df_err)

            df_top_err = df_err.head(num_of_features)
            all_sets_explaining_features[record_idx] = []
            shap_values_all_features = [[] for num in range(num_of_features)]

            backgroungd_set = self.get_background_set(x_train, 200).values
            for i in range(num_of_features):
                self.counter = df_top_err.index[i]
                explainer = shap.KernelExplainer(self.func_predict_feature, backgroungd_set)
                shap_values = explainer.shap_values(record_to_explain, nsamples='auto')
                shap_values_all_features[i] = shap_values

            shap_values_all_features = np.fabs(shap_values_all_features)

            shap_values_all_features = pd.DataFrame(data=shap_values_all_features, columns=x_train.columns)
            highest_contributing_features = self.get_highest_shap_values(shap_values_all_features)

            for idx_explained_feature in range(num_of_features):
                set_explaining_features =[]
                for idx, row in highest_contributing_features.iterrows():
                    if idx[0] == idx_explained_feature:
                        set_explaining_features.append((idx[1], row[0]))
                explained_feature_index = df_top_err.index[idx_explained_feature]
                set_explaining_features.insert(0, (x_train.columns[explained_feature_index], -1))

                all_sets_explaining_features[record_idx].append(set_explaining_features)

            final_set_features = []
            final_set_items = []
            for item in sum(all_sets_explaining_features[record_idx], []):
                if item[0] not in final_set_features:
                    final_set_features.append(item[0])
                    final_set_items.append(item)

            if return_shap_values:
                all_sets_explaining_features[record_idx] = final_set_items
            else:
                all_sets_explaining_features[record_idx] = final_set_features

        return all_sets_explaining_features

In [ ]:
from google.colab import files
uploaded = files.upload()  # This will prompt you to select the file


Saving requirement.txt to requirement (3).txt


In [ ]:
!pip install -r requirement.txt


ERROR: Invalid requirement: '_ipyw_jlab_nb_ext_conf=0.1.0=py37_0' (from line 4 of requirement.txt)
Hint: = is not a valid operator. Did you mean == ?


In [ ]:
pip install shap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 538.2/538.2 kB 5.0 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade pandas

import pandas as pd

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 26.4 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.0.3
    Uninstalling pandas-2.0.3:
      Successfully uninstalled pandas-2.0.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.0.3, but you have pandas 2.2.1 which is incompatible.


In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/creditcard_org.csv'

Mounted at /content/drive


In [ ]:
df = pd.read_csv(file_path, delimiter=',')
df = df.drop(['Time'], axis=1)
for col in df.columns[:-1]:
    min_val = df[col].min()
    max_val = df[col].max()
    if min_val != max_val:
        df[col] = (df[col] - min_val) / (max_val - min_val)

df.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.935192,0.766490,0.881365,0.313023,0.763439,0.267669,0.266815,0.786444,0.475312,0.510600,...,0.561184,0.522992,0.663793,0.391253,0.585122,0.394557,0.418976,0.312697,0.005824,0
1,0.978542,0.770067,0.840298,0.271796,0.766120,0.262192,0.264875,0.786298,0.453981,0.505267,...,0.557840,0.480237,0.666938,0.336440,0.587290,0.446013,0.416345,0.313423,0.000105,0
2,0.935217,0.753118,0.868141,0.268766,0.762329,0.281122,0.270177,0.788042,0.410603,0.513018,...,0.565477,0.546030,0.678939,0.289354,0.559515,0.402727,0.415489,0.311911,0.014739,0
3,0.941878,0.765304,0.868484,0.213661,0.765647,0.275559,0.266803,0.789434,0.414999,0.507585,...,0.559734,0.510277,0.662607,0.223826,0.614245,0.389197,0.417669,0.314371,0.004807,0
4,0.938617,0.776520,0.864251,0.269796,0.762975,0.263984,0.268968,0.782484,0.490950,0.524303,...,0.561327,0.547271,0.663392,0.401270,0.566343,0.507497,0.420561,0.317490,0.002724,0


In [ ]:
X = df.iloc[:,:-1]
y = df.iloc[:, -1]

print('x shape:', X.shape)
print('y shape:', y.shape)
print(y.value_counts())

train_idx = y[y==0].index.values
test_idx = y[y==1].index.values

X_train = X.iloc[train_idx]
y_train = y[train_idx]

X_test = X.iloc[test_idx]
y_test = y[test_idx]

x shape: (284807, 29)
y shape: (284807,)
0    284315
1       492
Name: Class, dtype: int64


In [ ]:
exp_model = ExplainAnomaliesUsingSHAP(num_anomalies_to_explain=10)

In [ ]:
all_sets_explaining_features = exp_model.explain_unsupervised_data(x_train=X_train,
                                                                   x_explain=X_test,
                                                                   return_shap_values=True)

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 29)]              0         
                                                                 
 dense (Dense)               (None, 14)                420       
                                                                 
 dense_1 (Dense)             (None, 7)                 105       
                                                                 
 dense_2 (Dense)             (None, 14)                112       
                                                                 
 dense_3 (Dense)             (None, 29)                435       
                                                                 
Total params: 1072 (4.19 KB)
Trainable params: 1072 (4.19 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/1000
3999/3999 - 8s -

In [ ]:
all_sets_explaining_features

{154684: [('V10', -1),
  ('V4', 0.019699173556512555),
  ('V15', 0.005698019149961078),
  ('V1', 0.005517353272361407),
  ('V9', 0.004791376112426258),
  ('V12', 0.003597360900956197),
  ('V13', 0.002893045270812303),
  ('V3', 0.0022438429719114764),
  ('V17', 0.0019185890629124765),
  ('V8', -1),
  ('V18', -1)],
 154587: [('V10', -1),
  ('V4', 0.019233363779261756),
  ('V15', 0.007069157135104719),
  ('V1', 0.0056041964880656),
  ('V9', 0.004753800902058404),
  ('V12', 0.0036629234981874),
  ('V13', 0.002602160338453114),
  ('V3', 0.0019574872597683327),
  ('V17', 0.0008238089362939923),
  ('V8', -1),
  ('V18', -1)],
 8296: [('V11', -1),
  ('V4', 0.01988954443068765),
  ('V12', 0.010147754563736996),
  ('V15', 0.003669847286542075),
  ('V17', 0.0034732375853960317),
  ('V9', 0.002920258368745675),
  ('V14', 0.0025669554789996196),
  ('V24', 0.0005773148535418071)],
 150644: [('V12', -1),
  ('V4', 0.01404339623154024),
  ('V15', 0.007677237424625814),
  ('V17', 0.0038480303535192487),


In [ ]:
from google.colab import drive

# Mount Google Drive
file_path_1 = '/content/drive/MyDrive/train_set_colab.csv'
file_path_2 = '/content/drive/MyDrive/test_set_colab.csv'

In [ ]:
x1 = pd.read_csv(file_path_1, delimiter=',')
x1.head()
numeric_attributes = ['tmpc', 'dwpc','relh', 'wspeedkm','pressure','vsbykm', 'feelc']
df1= x1[numeric_attributes]
for col in df1.columns:
    min_val = df1[col].min()
    max_val = df1[col].max()
    if min_val != max_val:
        df1[col] = (df1[col] - min_val) / (max_val - min_val)


In [ ]:
x2 = pd.read_csv(file_path_2, delimiter=',')
x2.head()
numeric_attributes = ['tmpc', 'dwpc','relh', 'wspeedkm','pressure','vsbykm', 'feelc']
df2= x2[numeric_attributes]

for col in df2.columns:
    min_val = df2[col].min()
    max_val = df2[col].max()
    if min_val != max_val:
        df2[col] = (df2[col] - min_val) / (max_val - min_val)





In [ ]:
print(df1.head())

       tmpc  dwpc  relh  wspeedkm  pressure    vsbykm     feelc
0  0.242424   0.4   1.0       0.0  0.347458  0.033557  0.262374
1  0.242424   0.4   1.0       0.0  0.347458  0.033557  0.262374
2  0.242424   0.4   1.0       0.0  0.347458  0.033557  0.262374
3  0.242424   0.4   1.0       0.0  0.341102  0.033557  0.262374
4  0.242424   0.4   1.0       0.0  0.328390  0.033557  0.262374


In [ ]:
print(df2.head())

       tmpc  dwpc      relh  wspeedkm  pressure    vsbykm     feelc
0  0.606061  0.80  0.917911       0.0  0.353846  0.125899  0.596026
1  0.575758  0.80  1.000000       0.0  0.353846  0.125899  0.566225
2  0.575758  0.80  1.000000       0.0  0.353846  0.125899  0.566225
3  0.575758  0.80  1.000000       0.0  0.353846  0.125899  0.566225
4  0.545455  0.76  1.000000       0.0  0.307692  0.125899  0.536424


In [ ]:
exp_model_1 = ExplainAnomaliesUsingSHAP(num_anomalies_to_explain=10)

In [ ]:
all_sets_explaining_features_1 = exp_model_1.explain_unsupervised_data(x_train=df1,
                                                                   x_explain=df2,autoencoder=None,
                                                                   return_shap_values=False)

Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_5 (InputLayer)        [(None, 7)]               0         
                                                                 
 dense_16 (Dense)            (None, 7)                 56        
                                                                 
 dense_17 (Dense)            (None, 3)                 24        
                                                                 
 dense_18 (Dense)            (None, 1)                 4         
                                                                 
 dense_19 (Dense)            (None, 7)                 14        
                                                                 
Total params: 98 (392.00 Byte)
Trainable params: 98 (392.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/1000
1110/1110 

AttributeError: 'Series' object has no attribute 'iteritems'

In [ ]:
all_sets_explaining_features_1

{13914: [('wspeedkm', -1),
  ('vsbykm', 0.007098272867012948),
  ('tmpc', 0.006262662306095336),
  ('relh', 0.003726427532244644)],
 435: [('wspeedkm', -1),
  ('tmpc', 0.009218497038974096),
  ('vsbykm', 0.007644969969056554),
  ('relh', 0.0038924919438237983)],
 16753: [('dwpc', -1),
  ('tmpc', 0.01578379316982764),
  ('relh', 0.0053242470235341476),
  ('vsbykm', 0.004704886069964917),
  ('feelc', -1)],
 16751: [('dwpc', -1), ('tmpc', 0.014109826431742663), ('feelc', -1)],
 16754: [('dwpc', -1),
  ('tmpc', 0.015793097826341782),
  ('relh', 0.00690178104738398),
  ('vsbykm', 0.004699499463041742),
  ('feelc', -1)],
 16755: [('dwpc', -1),
  ('tmpc', 0.015796642590136765),
  ('relh', 0.006901924904258467),
  ('vsbykm', 0.00470008308795244),
  ('feelc', -1)],
 16756: [('dwpc', -1),
  ('tmpc', 0.015796642590136765),
  ('relh', 0.006901924904258467),
  ('vsbykm', 0.00470008308795244),
  ('feelc', -1)],
 16757: [('dwpc', -1),
  ('tmpc', 0.01579902961743736),
  ('relh', 0.006901721853940311),